# Robust optimization for RCPSP

In this notebook, we demonstrate different ways to perform multi-objective optimisation for RCPSP (single and multi mode). 

## Scenario generation

To perform robust optimization, it is necessary to have either stochastic models of the environment or to have many scenarios related to the same problem instance.

For RCPSP, we can generate scenarios based on an initial problem instance scenario and using assumption on the distribution of the activity durations and resource availability. 

Details in this notebook: 
[RCPSP scenario generation](./Robustness-%20Creation%20of%20scenarios%20and%20robust%20static%20scenario.ipynb)

Once many scenarios are generated, they can support robust optimization in several ways:
* Optimize in a static way a chosen scenario from the sampled scenarios (e.g. worst-case, best-case or mean scenario)
* Optimize in a static way but using evaluating schedules by averaging over several scenarios for each objective
* Evaluate the quality of a schedule by keeping some scenarios for testing only

Those topics are illustrated in thie notebook


## Solving a selected instance to ensure robustness (e.g. solving the worst-case scenario)

With this approach, we optimize the schedule for a single scenario: the worst-case scenario, assuming that if a schedule is good for the worst case scenario, it will be robust to many events and will not break when exxectued under other scenarios

Details in this notebook: 
[RCPSP solving the worst-case scenario](./RCPSP%20Solving%20the%20worst-case%20scenario.ipynb)

## Evaluating RCPSP solutions using mean over several scenarios

With this approach, solutions to RCPSP are nt evaluated on a single scenario (instance) but average over many. So if makespan is an objective, the makespan of the solution will be the average over all makespans obtained on the different training scenarios.

Details in this notebook: 
[RCPSP solving using mean over many scenario](./RCPSP%20solving%20using%20means%20over%20many%20scenarios.ipynb)

## Using a robustness metric as part of the optimization

With this approach, we add an extra objective to the problem that relates to its robustness. This extra objective is obviously domain specific and in the case of RCPSP we have defined the resource reserve over time as a measure of robustness. Schedule with a lot of resource reserve can be considered as more robust than schedules with low reserves.

Once the extra objective defined, we can solve the problem using any multi-objective approach.

For this example, we use a simple aggregation of objectives in order to get a single solution after the run. Alternatively, we could have used a Pareto approach. We chose not to do so as in that case, we would have had to pick a solution from the Pareto front.

Details in this notebook: 
[RCPSP solving using aggregation of many objectives](./MO-aggregation-rcpsp.ipynb)

## Comparison of results from the different approaches

We have presented several approaches to perform robust optimization, each with clear drawbacks or advantages. We now perform a comparison of the solutions obtained. To do so, we make use of a test scenario set, not seen by the solver.

Note that the training scenarios differ across solvers. It does not matter much in this analysis, but the quality and fairness of the comparison could be improved by being careful with this.

### Generating the test scenarios

In [ ]:
import sys, os
sys.path.append('../')
from discrete_optimization.rcpsp.rcpsp_parser import parse_file, files_available, get_data_available
from discrete_optimization.rcpsp.rcpsp_model import RCPSPModel, SingleModeRCPSPModel, MultiModeRCPSPModel, \
RCPSPSolution, plt, create_poisson_laws, UncertainRCPSPModel, Aggreg_RCPSPModel, MethodAggregating, \
MethodBaseRobustification, MethodRobustification
from discrete_optimization.generic_tools.do_problem import ObjectiveHandling, MethodAggregating, \
BaseMethodAggregating, ParamsObjectiveFunction, ModeOptim, build_evaluate_function_aggregated
from discrete_optimization.rcpsp.rcpsp_utils import plot_resource_individual_gantt, plot_ressource_view
from discrete_optimization.generic_tools.result_storage.result_storage import ResultStorage
from discrete_optimization.generic_tools.result_storage.resultcomparator import ResultComparator
import random
import pickle

problem_instance = 'j301_1.sm'

files = get_data_available()
files = [f for f in files if problem_instance in f]
file_path = files[0]
rcpsp_model: RCPSPModel = parse_file(file_path)
if isinstance(rcpsp_model, MultiModeRCPSPModel):
    rcpsp_model.set_fixed_modes([1 for i in range(rcpsp_model.n_jobs)])
poisson_laws = create_poisson_laws(rcpsp_model, 
                                   range_around_mean_duration=2, 
                                   range_around_mean_resource=1,
                                   do_uncertain_duration=True, 
                                   do_uncertain_resource=False)
uncertain_model: UncertainRCPSPModel = UncertainRCPSPModel(base_rcpsp_model=rcpsp_model,
                                                           poisson_laws=poisson_laws)

many_random_instance = [uncertain_model.create_rcpsp_model(method_robustification=
                                                           MethodRobustification(MethodBaseRobustification.SAMPLE))
                        for i in range(300)]
len_random_instance = len(many_random_instance)

random.shuffle(many_random_instance)
proportion_train = 0.8
len_train = int(proportion_train*len_random_instance)
train_data = many_random_instance[:len_train]
test_data = many_random_instance[len_train:]


In [ ]:
# RETRIEVE SAVED SOLUTIONS
sol_to_retrieve = [problem_instance+'_worst-case_mip_grb.pkl', problem_instance+'_worst-case_ga.pkl', problem_instance+'_mean_over_scenario_ga.pkl', problem_instance+'_aggregation_hc_single_sol.pkl']

all_result_storages = []
result_storage_names = sol_to_retrieve
objectives_str = ['makespan', 'mean_resource_reserve']
objective_weights = [-1, 1]
# objectives_str = ['makespan']
# objective_weights = [-1]
test_problems = test_data

# CREATE 1 RESULT STORAGE PER SOLUTION 
params_objective_function = ParamsObjectiveFunction(objective_handling=ObjectiveHandling.MULTI_OBJ,
                                                    objectives=objectives_str,
                                                    weights=objective_weights,
                                                    sense_function=ModeOptim.MAXIMIZATION)
evaluate_sol, _ = build_evaluate_function_aggregated(problem=rcpsp_model,
                                                     params_objective_function=
                                                     params_objective_function)

for sol in sol_to_retrieve:
    print('sol: ', sol)
    thefile = open(sol, 'rb')   
    the_sol = pickle.load(thefile)
    # convert to rs
    sols = []
    s_pure_int = [i for i in the_sol.rcpsp_permutation]
    kwargs = {'rcpsp_permutation': s_pure_int, 'problem': rcpsp_model}
    problem_sol = rcpsp_model.get_solution_type()(**kwargs)
    fits = evaluate_sol(problem_sol)
    sols.append((problem_sol, fits))
    rs = ResultStorage(list_solution_fits=sols, best_solution=None)
    
    print('length or rs: ', len(rs.list_solution_fits))
    all_result_storages.append(rs)
    
# CREATE THE RESULT COMPARATOR
result_comparator = ResultComparator(list_result_storage=all_result_storages,
                                     result_storage_names=sol_to_retrieve,
                                     objectives_str=objectives_str,
                                     objective_weights=objective_weights,
                                     test_problems=test_problems)

### Plot distribution of makespan over test instances

In [ ]:
result_comparator.plot_distribution_for_objective(objective_str=objectives_str[0])
plt.show()

### Plot distribution of resource reserve over test instances

In [ ]:
result_comparator.plot_distribution_for_objective(objective_str=objectives_str[1])
plt.show()